# <u>Naive Bayes (NB)</u>

## Topics

* [1. Core Idea](#idea)
* [2. Key assumption (the "naive" part)](#naive)
* [3. How it models features](#models)
    * [3.1 Numerical Features](#numerical)
    * [3.2 Categorical Features](#categorical)
* [4. Laplace smoothing](#smoothing)
* [5. Properties & intuition](#properties)
* [6. Typical application](#apply)
* [7. Naive Bayes implementation (Spam mails)](#implement)
* [8. Naive Bayes library](#library)


In [1]:
import numpy as np # for rnadom number, linear algebra and general mathematic operations
from matplotlib import pyplot as plt # for plotting in 2d
import plotly.express as px # for plotting in 3d
import plotly.graph_objects as go # for plotting in 3d
from scipy.stats import norm # for normal pdf

print("Setup complete")

Setup complete


<a class="anchor" id="idea"></a>
# 1. Core idea

Naive Bayes is a generative, probabilistic classifier that computes the probability of each class using Bayes theorem:
$$
p(y=k \mid x)=\frac{p(x \mid y=k)p(y=k)}{p(x)}=\frac{p(x \mid y=k)\pi_k}{\sum_{j=1}^g p(x \mid y=j)p(y=j)}
$$

- $p(y=k)$: prior (class frequency)
- $p(x \mid y=k)$: likelihood
- Predict class with highest posterior probability

<a class="anchor" id="naive"></a>
# 2. Key assumption (the "naive" part)

NB assumes conditional independence of features given the class:

$$
\begin{align*}
p(x \mid y=k)&=p((x_1,\ldots,x_p) \mid y=k)\\
&=\prod_{j=1}^p p(x_j \mid y=k) \\
&= \prod_{j=1}^p \frac{1}{\sqrt{2\pi\sigma_{kj}^2}}\exp\left(-\frac{1}{2\sigma_{kj}^2}(x_j - \mu_{kj})^2\right)\\
&= \prod_{j=1}^p (2\pi\sigma_{kj}^2)^{-1/2}\exp\left(-\frac{1}{2\sigma_{kj}^2}(x_j - \mu_{kj})^2\right)
\end{align*}
$$

&#128073; This simplifies modeling a lot:
- Instead of one complex joint distribution
- Only estimate univariate distributions $p(x_j \mid y=k)$ per feature are needed

<a class="anchor" id="models"></a>
# 3. How it models features

<a class="anchor" id="numerical"></a>
## 3.1 Numerical Features



- Modeled with Gaussian distributions

$$
\begin{align*}
p(y=k \mid x)&=\frac{p(x \mid y=k)\pi_k}{p(x)} \\
&\propto p(x \mid y=k)\pi_k \\
&\overset{\text{independent}}{=} \prod_{i=1}^p p(x_j \mid y=k)\pi_k \hspace{1 mm} \mid \hspace{1 mm} \text{ difference between QDA and BA is conditional independence assumption} \\
&= \pi_k \prod_{i=1}^p p(x_j \mid y=k) \\
&= \pi_k \prod_{j=1}^p (2\pi\sigma_{kj}^2)^{-1/2}\exp\left(-\frac{1}{2\sigma_{kj}^2}(x_j - \mu_{kj})^2\right) \\
&= \log \left(\pi_k \prod_{j=1}^p (2\pi\sigma_{kj}^2)^{-1/2}\exp\left(-\frac{1}{2\sigma_{kj}^2}(x_j - \mu_{kj})^2\right)\right) \\

&= \log(\pi_k) + \log\left(\prod_{j=1}^p (2\pi\sigma_{kj}^2)^{-1/2}\exp\left(-\frac{1}{2\sigma_{kj}^2}(x_j - \mu_{kj})^2\right)\right) \\

&= \log(\pi_k) + \sum_{j=1}^p \log\left( (2\pi\sigma_{kj}^2)^{-1/2}\exp\left(-\frac{1}{2\sigma_{kj}^2}(x_j - \mu_{kj})^2\right)\right) \\

&= \log(\pi_k) + \sum_{j=1}^p \left( \log((2\pi\sigma_{kj}^2)^{-1/2}) + \log\left(\exp\left(-\frac{1}{2\sigma_{kj}^2}(x_j - \mu_{kj})^2\right)\right)\right) \\

&= \log(\pi_k) + \sum_{j=1}^p \left( -\frac{1}{2} \log(2\pi\sigma_{kj}^2) -\frac{1}{2\sigma_{kj}^2}(x_j - \mu_{kj})^2\right) \\

&= \log(\pi_k) + \sum_{j=1}^p \left( -\frac{1}{2} \log(2\pi\sigma_{kj}^2) -\frac{1}{2\sigma_{kj}^2}(x_j - \mu_{kj})^2\right) \\

&= \log(\pi_k) -\frac{1}{2} \sum_{j=1}^p \left(\log(2\pi\sigma_{kj}^2) -\frac{(x_j - \mu_{kj})^2}{\sigma_{kj}^2}\right) \\

&= \log(\pi_k) -\frac{1}{2} \sum_{j=1}^p \log(2\pi\sigma_{kj}^2) -\frac{1}{2} \sum_{j=1}^p \left(\frac{(x_j - \mu_{kj})^2}{\sigma_{kj}^2}\right) \\

&= \log(\pi_k) -\frac{1}{2} \sum_{j=1}^p \log(2\pi\sigma_{kj}^2) -\frac{1}{2} \sum_{j=1}^p \left(\frac{x_j^2}{\sigma_{kj}^2} - \frac{2x_j\mu_{kj}}{\sigma_{kj}^2} + \frac{\mu_{kj}^2}{\sigma_{kj}^2}\right) \\

&= \log(\pi_k) -\frac{1}{2} \sum_{j=1}^p \log(2\pi\sigma_{kj}^2)-\frac{1}{2} \sum_{j=1}^p \frac{x_j^2}{\sigma_{kj}^2} +\frac{1}{2} \sum_{j=1}^p \frac{2x_j\mu_{kj}}{\sigma_{kj}^2} -\frac{1}{2} \sum_{j=1}^p \frac{\mu_{kj}^2}{\sigma_{kj}^2} \\

&= \underbrace{\log(\pi_k) -\frac{1}{2} \sum_{j=1}^p \log(2\pi\sigma_{kj}^2) -\frac{1}{2} \sum_{j=1}^p \frac{\mu_{kj}^2}{\sigma_{kj}^2}}_{w_{0k}} + \underbrace{\sum_{j=1}^p \frac{\mu_{kj}}{\sigma_{kj}^2}x_j}_{w_k} - \underbrace{\frac{1}{2} \sum_{j=1}^p \frac{1}{\sigma_{kj}^2}x_j^2}_{W_k}=: f_k(x)   \\

\end{align*}
$$

- Group terms:
    - Quadratic in $x_j: x_j^2$
    - Linear in $x_j: x_j$
    - Constant terms


- Estimate mean and variance per feature & class

Class priors: $$\hat{\pi}_k=\frac{n_k}{n}$$

- $n$ is total number of observations
- $n_k$ number observation in class $k$

Class means for each feature $j$ and class $k$: $$\hat{\mu}_{kj}=\frac{1}{n_k} \sum_{i:y^{(i)}=k} x^{(ij)}$$


Class variance: $$\hat{\sigma}_{kj}^2=\frac{1}{n_k-1}  \sum_{i:y^{(i)}=k} (x^{(ij)}-\hat{\mu}_{kj})^2$$

- Equivalent to QDA with diagonal covariance matrix

&#128073; These define: $p(x_j \mid y=k)=\mathcal{N}(\mu_{kj},\sigma_{kj}^2)$

### Decision boundary

**For two classes $k$ and $l$ the boundary is where:**

$$
f_k(x) = f_l(x)
$$

**So set to zero and solve for an axis to get quadratic descision boundary:**

$$
f_k(x) - f_l(x)=0
$$

<a class="anchor" id="categorical"></a>
## 3.2 Categorical Features

- Modeled with categorical (frequency-based) distributions for $p(x_j \mid y=k)$
- Estimate probabilities $p_{kjm}$ by counting occurrences

$$
p(x_j \mid y=k) = \prod_m p_{kjm}^{[x_j=m]} \overset{\text{independence}}\Longrightarrow p(x \mid y=k) = \prod_{j=1}^p \underbrace{\prod_m p_{kjm}^{[x_j=m]}}_{p(x_j \mid y=k)}
$$

- $p_{kjm}=p(x_j=m \mid y=k)$ = Probability that in class $k$ the $j$-th feature $x_j$ has value $x_j=m$ 

- More precisely $p_{kjm}=\frac{n_{kjm}}{n_k} \Rightarrow p_{kjm}^{[x_j=m]}=(\frac{n_{kjm}}{n_k})^{[x_j=m]}$ with $n_{kjm}$ being the number of observation in class $k$ where the $j$-th feature $x_j$ has value $x_j=m$ and $n_k$ the total counts in class $k$

- $[x_j = m]=\begin{cases} 1 &\text{if }x_j = m \\ 0 &\text{otherwise}   \end{cases} \hspace{1 mm}$ keeps only the probability for the actual value

**Example**

$$
\begin{array}{}
\hline
\text{Level} & \text{Gender} & \text{Survived} \\
\hline
2nd & \text{male} & \text{no} \\
1st & \text{male}  & \text{yes}  \\
3rd & \text{female}  & \text{yes} \\
1st & \text{female}  & \text{yes}  \\
2nd & \text{female}  & \text{yes} \\
3rd & \text{female}  & \text{no} \\
\end{array}
$$

$$
\begin{align*}
p(x_{\text{gender}} \mid y = \text{yes})&=\prod_{m \in \text{male,female}} p_{\text{yes},\text{sex},m}^{[x_\text{gender}=m]} \\
&=p_{\text{yes},\text{sex},\text{male}}^{[x_\text{gender}=\text{male}]} \cdot p_{\text{yes},\text{sex},\text{female}}^{[x_\text{gender}=\text{female}]}  \\

&=\frac{1}{4}^{[x_\text{gender}=\text{male}]} \cdot \frac{3}{4}^{[x_\text{gender}=\text{female}]} = \begin{cases} \frac{1}{4} &\text{if }x_\text{gender} = \text{male} \\  \frac{3}{4} &\text{if }x_\text{gender} = \text{female}  \end{cases}
\end{align*}
$$

For a full observation (e.g., Level + Gender):

$$
\begin{align*}
p((x_\text{Level},x_\text{Gender}) \mid y=\text{yes}) &\overset{\text{independence}}{=} p(x_\text{Level} \mid y=\text{yes}) \cdot p(x_\text{Gender} \mid y=\text{yes}) \\
&=  \prod_m p_{\text{yes},\text{Level},m}^{[x_\text{Level}=m]} \cdot \prod_m p_{\text{yes},\text{Gender},m}^{[x_\text{Gender}=m]} \\
\end{align*}
$$



<a class="anchor" id="smoothing"></a>
# 4. Laplace smoothing

## &#10071; Problem:

- If a feature value never appears in a class $\rightarrow$ probability = 0
- This makes the whole product = 0

Example: 

$$
\begin{array}{}
\hline
\text{Level} & \text{Gender} & \text{Survived} \\
\hline
2nd & \text{male} & \text{no} \\
1st & \text{male}  & \text{yes}  \\
3rd & \text{female}  & \text{yes} \\
1st & \text{female}  & \text{yes}  \\
2nd & \text{female}  & \text{yes} \\
3rd & \text{female}  & \text{no} \\
\end{array}
$$

We want the posterior $p(y=\text{no} \mid (x_\text{Level}=1st,x_\text{Gender}=\text{male}))$ so the probability of not surviving as a male in the first level.

$$
\begin{align*}
p(y=\text{no} \mid (x_\text{Level}=1st,x_\text{Gender}=\text{male}))&=\frac{p((x_\text{Level}=1st,x_\text{Gender}=\text{male}) \mid y=\text{no}) \cdot p(y=\text{no})}{p(x_\text{Level}=1st,x_\text{Gender}=\text{male})} \\

&=\frac{p((x_\text{Level}=1st,x_\text{Gender}=\text{male}) \mid y=\text{no}) \cdot p(y=\text{no})}{\sum_{j \in \{\text{yes},\text{no}\}}p((x_\text{Level}=1st,x_\text{Gender}=\text{male})\mid y=j) \cdot p(y=j)} \\

&\overset{\text{independence}}{=}\frac{p(x_\text{Level}=1st\mid y=\text{no}) \cdot p(x_\text{Gender}=\text{male}\mid y=\text{no}) \cdot p(y=\text{no})}{\sum_{j \in \{\text{yes},\text{no}\}}p(x_\text{Level}=1st\mid y=j) \cdot p(x_\text{Gender}=\text{male}\mid y=j) \cdot p(y=j)} \\

&\text{Problem: Value 1st of feature Level never appears in class no} \\

&=\frac{0 \cdot p(x_\text{Gender}=\text{male}\mid y=\text{no}) \cdot p(y=\text{no})}{\sum_{j \in \{\text{yes},\text{no}\}}p(x_\text{Level}=1st\mid y=j) \cdot p(x_\text{Gender}=\text{male}\mid y=j) \cdot p(y=j)}=0 \\


\end{align*}
$$


## &#129504; Solution:

- Add a small constant $\alpha > 0$ (usually $\alpha=1$):


$$
p(x_j=m \mid y=k)=\frac{n_{kjm} + \alpha}{n_k + \alpha M_j} \\

p(x_j \mid y=k)=\prod_m (\frac{n_{kjm} + \alpha}{n_k + \alpha M_j})^{[x_j=m]}



$$

- $M_j$ is number of possible values of $x_j$

<a class="anchor" id="properties"></a>
# 5. Properties & intuition

- Simple and fast
- Works well even with high-dimensional data
- Handles mixed data types (numerical + categorical)
- Despite wrong independence assumption $\rightarrow$ often performs surprisingly well

<a class="anchor" id="apply"></a>
# 6. Typical application

Classic example: spam filtering

- Features = word counts
- Assumes words occur independently (not true, but still effective)

<a class="anchor" id="implement"></a>
# 7. Naive Bayes implementation (Spam mails)

<a class="anchor" id="library"></a>
# 8. Naive Bayes library